### [GENDT] Automobile - Silver Layer

Import libs and start spark context

In [ ]:
import os, sys
from datetime import datetime

sys.path.append(os.path.join(os.getcwd(), "/home/jovyan/work/src/core"))

import spark_session
import pyspark.sql.functions as F
from pyspark.sql import Window

In [2]:
spark = spark_session.build_spark()

Start variables and read table

In [3]:
origin_table = 'brz_gendt.automobile'

target_schema = 'svr_gendt'
target_table = 'automobile'

In [4]:
df_automobile = spark_session.read_table(spark, origin_table)

Processing and cleaning data

In [5]:
# Remove records where value fields are null

cols_to_check = [col for col in df_automobile.columns if col not in ['name', 'model_year', 'origin', 'ingestion_datetime', 'ingestion_file']]

df_automobile = df_automobile.dropna(subset=cols_to_check)

In [6]:
# Select the most recent occurance of the record

df_automobile = df_automobile.dropDuplicates()

window_spec = Window.partitionBy('name').orderBy(F.desc('ingestion_datetime'))

df_automobile = df_automobile.withColumn(
    'rn', 
    F.row_number().over(window_spec)
).filter(F.col('rn') == 1).drop('rn')

Structure dataframe and write

In [7]:
df_automobile = df_automobile.select(
    F.split(F.col("name"), " ").getItem(0).alias('s_brand'),
    F.regexp_replace(F.col('name'), r"^\w+\s*", "").alias('s_model'),
    F.col('mpg').cast('decimal(10,2)').alias('v_miles_per_galoon'),
    F.col('cylinders').cast('decimal(10,2)').alias('v_cylinders'),
    F.col('horsepower').cast('decimal(10,2)').alias('v_horsepower'),
    F.col('weight').cast('decimal(10,2)').alias('v_weight'),
    F.col('acceleration').cast('decimal(10,2)').alias('v_acceleration'),
    F.col('model_year').cast('integer').alias('n_model_year'),
    F.col('origin').cast('string').alias('s_country_origin'),
    F.lit(datetime.now()).cast('timestamp').alias('d_slv_processing'),
    F.col('ingestion_datetime').cast('timestamp').alias('d_brz_ingestion'),
    F.col('ingestion_file').cast('string').alias('s_file_ingestion'),
)

In [8]:
spark_session.run_sql(f'CREATE SCHEMA IF NOT EXISTS {target_schema};')

CREATE SCHEMA


In [9]:
spark_session.write_table(df_automobile, target_schema, target_table, 'overwrite')

In [10]:
spark.stop()